# 03 | Protocol đánh giá khoá của V6 (nguồn số liệu công bố)

18 script trong `notebooks/v6/protocol/` chạy theo số thứ tự. Tham số dò trên DEV được đóng băng kèm SHA-256 trước khi chạm TEST. Đầu ra ghi vào `results/v6/`: bảng tổng hợp được commit, dự đoán từng cửa sổ thì không.

| Cần có trước | Từ đâu |
|---|---|
| `models/multidomain/healthsense_af_v6c_stacking.pkl` | notebook `02_train_stacking` với `OVERWRITE_FROZEN = True`, hoặc bản chuẩn đối chiếu `data/MANIFEST.csv` |
| `data/features/v6/`, `data/splits/` | notebook `00_extract_features_and_splits` |

Chạy lại được từng bước. Bảng kết quả hiện ở ô cuối.

In [ ]:
import os, sys, time, subprocess, glob
from pathlib import Path
ROOT = Path.cwd()
while not (ROOT / "src" / "healthsense_ml").exists():
    if ROOT.parent == ROOT: raise RuntimeError("Không tìm thấy gốc repo")
    ROOT = ROOT.parent
os.chdir(ROOT); sys.path.insert(0, str(ROOT / "src"))
print("Repo root:", ROOT)

def run(script, *args):
    """Chạy một script từ gốc repo bằng đúng Python của kernel; in log; dừng nếu lỗi."""
    t0 = time.time()
    print(f"\n$ python {script} {' '.join(args)}", flush=True)
    r = subprocess.run([sys.executable, script, *args], cwd=ROOT, text=True, encoding="utf-8", errors="replace",
                       capture_output=True, env={**os.environ, "PYTHONIOENCODING": "utf-8", "PYTHONPATH": "src"})
    print(r.stdout[-4000:])
    if r.returncode:
        print(r.stderr[-4000:]); raise RuntimeError(f"{script} lỗi, mã {r.returncode}")
    print(f"  xong trong {time.time()-t0:.0f}s")

In [ ]:
STEPS = sorted(glob.glob("notebooks/v6/protocol/[0-9][0-9]_*.py"))
assert len(STEPS) == 18, STEPS
for s in STEPS: print(s)

## Bước 01 → 03: dự đoán khoá, kiểm tra thứ tự thời gian, verified streams

In [ ]:
for s in STEPS[0:3]: run(s)

## Bước 04 → 07: dò luật cảnh báo trên DEV, chấm trên TEST

In [ ]:
for s in STEPS[3:7]: run(s)

## Bước 08 → 10: hiệu chỉnh Platt và khoảng tin cậy

In [ ]:
for s in STEPS[7:10]: run(s)

## Bước 11 → 14: LODO, ablation PPG_AC, PAC/PVC

In [ ]:
for s in STEPS[10:14]: run(s)

## Bước 15 → 18: DeepBeat forensics và chốt freeze

In [ ]:
for s in STEPS[14:18]: run(s)

## Bảng kết quả

In [ ]:
import pandas as pd
for f in ["results/v6/global_rule_v2_reused_test_summary.csv", "results/v6/lodo_domain_generalization_summary.csv",
          "results/v6/calibration_summary.csv", "results/v6/pac_pvc_challenge_summary.csv", "results/v6/global_rule_v2_exact_subject_ci.csv"]:
    print("\n==", f); display(pd.read_csv(f).round(4))